# **Ejercicio 2: Entregable Final (videogames_reviews.csv)**

Vamos a trabajar con un dataframe que contiene información sobre reseñas de videojuegos. Cada fila contiene la información de una reseña, y las columnas son las siguientes:
- `Contenido`: texto de la reseña
- `Valoración`: Recomendado o No recomendado
- `Recomendado_binario`: 1 si la valoración es Recomendado, 0 si es No recomendado. **Es la respuesta correcta con la que evaluarás tu prompt en el paso 4**

El objetivo es extraer información valiosa de las reseñas de un dataset más "crudo" que el del caso de Airbnb que resolvimos en clase, con un **pipeline de dos pasos con LLMs**, y **medir** cómo de bien funciona.

1. **Paso 1: las 50 reseñas con el contenido más largo.** Esto ya os lo doy hecho más abajo.
2. **Paso 2: filtrado con un primer LLM.** Muchas de esas reseñas no aportan nada (texto repetido, spam, bromas). Instruye vía prompting a un LLM para que se quede solo con las que tienen información de la que se pueden extraer insights.
3. **Paso 3: extracción con un segundo LLM.** Sobre las reseñas ya filtradas, extrae información estructurada. Por ejemplo, un JSON como este:

    ```json
    {
      "Id": "{{id}}",
      "SentimientoGeneral": "Positivo | Negativo | Mixto",
      "AspectosPositivos": ["aspecto 1", "aspecto 2"],
      "AspectosNegativos": ["aspecto 1", "aspecto 2"],
      "Dificultad": "Demasiado fácil | Fácil | Equilibrado | Difícil | Demasiado difícil | No mencionado",
      "Recomendado": "1 | 0 (predicción del LLM)"
    }
    ```

    El esquema es libre, con **un mínimo de 3 campos**, pero tiene que incluir la **predicción de si la reseña recomienda el juego**.
4. **Paso 4: evaluación del prompt.** Compara la predicción del LLM con la columna `Recomendado_binario`: calcula el accuracy, muestra una tabla cruzada y analiza brevemente los fallos. **Este paso es evaluable.**

**Consideraciones:**

- El LLM **no puede recibir** las columnas `Valoración` ni `Recomendado_binario`: solo el texto de la reseña. Si se las pasas, la evaluación del paso 4 no vale.
- Usa un proveedor vía API (Groq, como en clase, u otro) y **justifica** el modelo elegido en cada paso. Puede ser el mismo en los dos, explicando por qué.
- Usa **salida estructurada** (JSON con esquema) y valida la respuesta, como hicimos en el caso de Airbnb.
- Ten en cuenta los límites del proveedor: tokens por minuto y por día, peticiones por minuto. **Justifica** el tamaño de los lotes, cada pausa entre llamadas y qué ocurre si se alcanza un límite antes de terminar.
- Estas reseñas son muy largas (unos 7.000 caracteres cada una). Calcula cuántos tokens vas a gastar antes de lanzar nada y compáralo con los límites por minuto y por día de tu modelo. Si decides recortar el texto en algún paso, justifícalo.
- **Consejo:** desarrolla y prueba tus prompts con 5 o 10 reseñas. Lanza las 50 solo cuando funcione, o agotarás la cuota diaria haciendo pruebas.
- Tienes un CSV de ejemplo de resultado final (`analisis_videojuegos_resultados.csv`) con otras entidades y otro formato, por si te sirve de inspiración.
- La API key **no se escribe en el notebook**: léela de una variable de entorno o pídela con `getpass`.

**Entregable:** este notebook resuelto y ejecutado, con los resultados visibles y la justificación escrita de tus decisiones.

In [25]:
%pip install groq

import os
import json
import time
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from groq import Groq, RateLimitError, BadRequestError
from pydantic import BaseModel, ConfigDict, Field, ValidationError # Para definir el esquema JSON de cada paso y validar la respuesta

groq_api_key = os.environ.get("GROQ_API_KEY")

client = Groq(api_key=groq_api_key, max_retries=0)

# Un modelo para cada paso del pipeline
MODELO_FILTRO = "openai/gpt-oss-20b"      # paso 2: tarea sencilla (relevante sí/no)
MODELO_EXTRACCION = "openai/gpt-oss-120b" # paso 3: tarea más compleja (varios campos + recomendación)

Note: you may need to restart the kernel to use updated packages.


In [26]:
data = pd.read_csv('videogames_reviews.csv')
data.info()
data.head()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Unnamed: 0           20000 non-null  int64
 1   Contenido            19712 non-null  str  
 2   Valoración           20000 non-null  str  
 3   Recomendado_binario  20000 non-null  int64
dtypes: int64(2), str(2)
memory usage: 625.1 KB


,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0


In [27]:
# Análisis exploratorio básico
print("Información básica del dataframe:")
print(f"Forma del dataframe: {data.shape}")
print(f"Columnas: {list(data.columns)}")
print("\nTipos de datos:")
print(data.dtypes)
print("\nPrimeras filas:")
data.head()

Información básica del dataframe:
Forma del dataframe: (20000, 4)
Columnas: ['Unnamed: 0', 'Contenido', 'Valoración', 'Recomendado_binario']

Tipos de datos:
Unnamed: 0             int64
Contenido                str
Valoración               str
Recomendado_binario    int64
dtype: object

Primeras filas:


,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0


In [28]:
# Análisis de valores nulos y únicos
print("Valores nulos por columna:")
print(data.isnull().sum())
print("\nValores únicos en 'Valoración':")
print(data['Valoración'].value_counts())
print("\nEstadísticas de la columna 'Recomendado_binario':")
print(data['Recomendado_binario'].value_counts())

Valores nulos por columna:
Unnamed: 0               0
Contenido              288
Valoración               0
Recomendado_binario      0
dtype: int64

Valores únicos en 'Valoración':
Valoración
No recomendado    10000
Recomendado       10000
Name: count, dtype: int64

Estadísticas de la columna 'Recomendado_binario':
Recomendado_binario
0    10000
1    10000
Name: count, dtype: int64


## **PASO 1**
Obtener los 50 comentarios con más texto para análisis posterior.

In [29]:
# Preprocesamiento simplificado: Top 50 comentarios con más texto
print("Preprocesando datos para obtener los 50 comentarios más largos...")

# Eliminar filas con contenido nulo
df_limpio = data.dropna(subset=['Contenido']).copy()
print(f"Comentarios válidos (sin nulos): {len(df_limpio)}")

# Calcular longitud de caracteres
df_limpio['longitud_caracteres'] = df_limpio['Contenido'].str.len()

# Seleccionar los 50 comentarios más largos
df_top_50 = df_limpio.nlargest(50, 'longitud_caracteres').reset_index(drop=True)

# Eliminar columnas innecesarias
if 'Unnamed: 0' in df_top_50.columns:
    df_top_50 = df_top_50.drop('Unnamed: 0', axis=1)

print(f"\nDataset final: {len(df_top_50)} comentarios")
print(f"Longitud promedio: {df_top_50['longitud_caracteres'].mean():.1f} caracteres")
print(f"Longitud mínima: {df_top_50['longitud_caracteres'].min()} caracteres")
print(f"Longitud máxima: {df_top_50['longitud_caracteres'].max()} caracteres")

print("\nDistribución de valoraciones en el top 50:")
print(df_top_50['Valoración'].value_counts())

# Mostrar el dataframe resultante
df_top_50

Preprocesando datos para obtener los 50 comentarios más largos...
Comentarios válidos (sin nulos): 19712

Dataset final: 50 comentarios
Longitud promedio: 7369.6 caracteres
Longitud mínima: 6885 caracteres
Longitud máxima: 7972 caracteres

Distribución de valoraciones en el top 50:
Valoración
No recomendado    29
Recomendado       21
Name: count, dtype: int64


,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
1,this was probably my first preorder i felt tha...,Recomendado,1,7662
2,oh well admittedly its difficult for to write ...,No recomendado,0,7638
3,oh well admittedly its difficult for to write ...,No recomendado,0,7638
4,i know many will handwave away any criticisms ...,No recomendado,0,7609
5,i had enormous expectations having put thousan...,No recomendado,0,7609
6,i finished the witcher 3 for the first time th...,Recomendado,1,7604
7,in any industry this will always happen as an ...,No recomendado,0,7590
8,now my favorite fromsoft game tldr here phenom...,Recomendado,1,7588
9,now my favorite fromsoft game tldr here phenom...,Recomendado,1,7588


## **PASO 2: Filtrado con un primer LLM**

Diseña el prompt, llama al modelo por lotes y quédate con un DataFrame reducido, sin las reseñas irrelevantes.

### Paso 2.1: preparar las reseñas para el filtro

1. Quitar los duplicados con pandas, no con el LLM
2. recortar el texto que ve el filtro (Ahorrar tokens? AI recomendación)

In [30]:
# 2.1 Preparar las reseñas para el filtro

# Duplicados exactos (decisión pendiente)
# True = se quitan con pandas antes del LLM (comparación exacta, sin gastar tokens)
# False = se dejan y el filtrado lo hace el LLM (como sugiere el enunciado)
QUITAR_DUPLICADOS = True

print("Duplicados exactos en el top 50:", df_top_50['Contenido'].duplicated().sum())

if QUITAR_DUPLICADOS:
    df_top = df_top_50.drop_duplicates(subset='Contenido').copy()
else:
    df_top = df_top_50.copy()

# El índice (0-49) será el ID de cada reseña: así cruzamos las respuestas del LLM con la fila original
df_top.index.name = 'id'
print("Reseñas que pasan al filtro:", len(df_top))

# Recorte del texto (decisión pendiente)
# None = se envía la reseña entera; un número = solo los primeros N caracteres (1 token ≈ 4 caracteres)
# MAX_CARACTERES_FILTRO = None        # paso 2
MAX_CARACTERES_FILTRO = 1500      # ≈ 375 tokens: descomentar para recortar en el filtro
MAX_CARACTERES_EXTRACCION = None    # paso 3
# MAX_CARACTERES_EXTRACCION = 4000  # ≈ 1.000 tokens: descomentar para recortar en la extracción

def recortar(texto, max_caracteres):
    """Devuelve el texto entero si max_caracteres es None; si no, solo los primeros max_caracteres."""
    if max_caracteres is None:
        return texto
    return texto[:max_caracteres]

# Comprobación: cómo empieza cada reseña (lo que verá el LLM, primeros 120 caracteres)
for i, fila in df_top.head(8).iterrows():
    print(f"[{i}] {recortar(fila['Contenido'], MAX_CARACTERES_FILTRO)[:120]}")




Duplicados exactos en el top 50: 12
Reseñas que pasan al filtro: 38
[0] suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii
[1] this was probably my first preorder i felt that if the game turned out to be the worst game i played or didnt even run 1
[2] oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty bia
[4] i know many will handwave away any criticisms of this game as salt or reply with git gud but as fan of fromsoft games si
[5] i had enormous expectations having put thousand hours into dark souls 3 alone sadly er is tainted by design decisions no
[6] i finished the witcher 3 for the first time this year and because of the interesting story told i got invested and start
[7] in any industry this will always happen as an example like witcher that developed by the same company what they have sho
[8] now my favorite fromsoft game tldr here phenomenal an

### Paso 2.2: esquema y prompt del filtro

In [31]:
# Esquema de la respuesta del filtro: una entrada por reseña
class ResenaFiltrada(BaseModel):
    model_config = ConfigDict(extra="forbid")  # no se admiten campos que no estén aquí

    id: int = Field(description="El ID que aparece en la cabecera [ID n] de la reseña")
    motivo: str = Field(description="Explicación breve, máximo 15 palabras, de por qué es o no relevante")
    relevante: bool = Field(description="true si aporta opiniones o información útil sobre el juego; false si no")

class LoteFiltro(BaseModel):
    model_config = ConfigDict(extra="forbid")
    resenas: list[ResenaFiltrada]

# Salida estructurada en modo estricto: el modelo solo puede generar JSON que cumpla el esquema
FORMATO_FILTRO = {
    "type": "json_schema",
    "json_schema": {"name": "lote_filtro", "strict": True, "schema": LoteFiltro.model_json_schema()},
}

SYSTEM_FILTRO = """Eres un moderador de contenido de una plataforma de reseñas de videojuegos.
Recibirás VARIAS reseñas, cada una con una cabecera [ID n]. El texto viene preprocesado: en minúsculas, sin signos de puntuación, y puede mezclar inglés y español.

Tu tarea: decidir si cada reseña es RELEVANTE para analizar opiniones sobre el juego.

Es RELEVANTE si contiene opiniones o información útil sobre el juego: aspectos positivos o negativos (jugabilidad, historia, gráficos, rendimiento, precio, dificultad...), la experiencia de juego o una valoración razonada. Lo es aunque sea informal o tenga alguna parte en broma.

NO es relevante si:
- Es spam o texto sin sentido (letras o palabras repetidas, como "suiiiii").
- Repite la misma frase o párrafo una y otra vez sin aportar opiniones.
- No habla del juego (publicidad, enlaces, temas ajenos).
- Es solo una broma, un meme o la letra de una canción, sin valoración del juego.

En caso de duda, márcala como relevante: es preferible pasar una reseña dudosa al siguiente paso que perder una útil.

Devuelve exactamente un elemento por reseña, con su mismo ID."""

# Si los duplicados no se quitan con pandas (QUITAR_DUPLICADOS = False en el 2.1), se le pide al LLM que los detecte
if not QUITAR_DUPLICADOS:
    SYSTEM_FILTRO += """

Además: si una reseña es una copia exacta o casi exacta de otra reseña de este mismo lote, márcala como NO relevante, con el motivo "duplicada del ID n", y conserva solo la de menor ID."""

# Comprobación: el JSON Schema que recibirá la API
LoteFiltro.model_json_schema()


{'$defs': {'ResenaFiltrada': {'additionalProperties': False,
   'properties': {'id': {'description': 'El ID que aparece en la cabecera [ID n] de la reseña',
     'title': 'Id',
     'type': 'integer'},
    'motivo': {'description': 'Explicación breve, máximo 15 palabras, de por qué es o no relevante',
     'title': 'Motivo',
     'type': 'string'},
    'relevante': {'description': 'true si aporta opiniones o información útil sobre el juego; false si no',
     'title': 'Relevante',
     'type': 'boolean'}},
   'required': ['id', 'motivo', 'relevante'],
   'title': 'ResenaFiltrada',
   'type': 'object'}},
 'additionalProperties': False,
 'properties': {'resenas': {'items': {'$ref': '#/$defs/ResenaFiltrada'},
   'title': 'Resenas',
   'type': 'array'}},
 'required': ['resenas'],
 'title': 'LoteFiltro',
 'type': 'object'}

### Paso 2.3: llamada al modelo y prueba con 2 reseñas

In [32]:
# 2.3 Reintentos (igual que en el caso Airbnb): si hay rate limit o la salida no es válida, espera y repite
# ejecuta otra función y, si falla, la repite (hasta 5 veces). Ante un límite de uso (429) espera lo que indica la API; ante una respuesta no válida, espera 1 s y vuelve a intentarlo.
def con_reintentos(funcion, *args, max_intentos=5, **kwargs):
    for intento in range(1, max_intentos + 1):
        try:
            return funcion(*args, **kwargs)
        except RateLimitError as error:
            espera = float(error.response.headers.get("retry-after", 2 ** intento)) + 1
            print(f"   429: esperamos {espera:.0f} s (intento {intento} de {max_intentos})")
            time.sleep(espera)
        except (ValidationError, ValueError, BadRequestError) as error:
            print(f"   Salida no válida: {str(error)[:120]} (intento {intento} de {max_intentos})")
            time.sleep(1)
    raise RuntimeError(f"No se ha podido completar tras {max_intentos} intentos")

# une varias reseñas en un solo mensaje, cada una con su cabecera `[ID n]` y separadas por `---`, aplicando el recorte si está activado.
def texto_lote(filas, max_caracteres):
    """Une varias reseñas en un solo texto, cada una con su cabecera [ID n]."""
    bloques = []                                   # lista vacía
    for i, fila in filas.iterrows():               # recorre cada reseña: i = su ID, fila = sus datos
        texto = recortar(fila['Contenido'], max_caracteres)   # su texto (recortado si toca, el interructor del principio)
        bloque = f"[ID {i}]\n{texto}"             # le pone la etiqueta delante
        bloques.append(bloque)                     # lo añade a la lista

    return "\n\n---\n\n".join(bloques)

# Envía el lote al modelo del filtro y comprueba que la respuesta no esté cortada, 
# cumpla el esquema y devuelva todos los IDs. Devuelve la decisión de cada reseña y los tokens usados.
def filtrar_lote(filas):
    """Envía un lote de reseñas al modelo del filtro. Devuelve (lista de ResenaFiltrada, usage)."""
    response = client.chat.completions.create(
        model=MODELO_FILTRO,
        messages=[
            {"role": "system", "content": SYSTEM_FILTRO},
            {"role": "user", "content": texto_lote(filas, MAX_CARACTERES_FILTRO)},
        ],
        temperature=0,                                   # clasificación: queremos siempre la misma respuesta
        reasoning_effort="low",                          # tarea sencilla: poco razonamiento, menos tokens
        max_completion_tokens=500 + 100 * len(filas),    # la salida crece con el tamaño del lote
        response_format=FORMATO_FILTRO,
    )

    # Si la respuesta se ha cortado por longitud, el JSON estará incompleto
    if response.choices[0].finish_reason == "length":
        raise ValueError("Respuesta cortada: sube max_completion_tokens o reduce el lote")

    # Validamos con Pydantic: si no cumple el esquema, salta un ValidationError
    lote = LoteFiltro.model_validate_json(response.choices[0].message.content)

    # Comprobamos que vuelven exactamente los IDs que hemos enviado
    ids_enviados = sorted(int(i) for i in filas.index)
    ids_recibidos = sorted(r.id for r in lote.resenas)
    if ids_enviados != ids_recibidos:
        raise ValueError(f"IDs enviados {ids_enviados} y recibidos {ids_recibidos} no coinciden")

    return lote.resenas, response.usage


# Prueba con 2 reseñas: la 0 (spam "suiii") y la 1 (reseña normal)
resenas, uso = con_reintentos(filtrar_lote, df_top.loc[[0, 1]])

for r in resenas:
    print(f"[{r.id}] relevante={r.relevante} · {r.motivo}")

print(f"\nTokens de entrada: {uso.prompt_tokens}")
print(f"Tokens de salida:  {uso.completion_tokens}")
print(f"Total:             {uso.total_tokens}  (≈ {uso.total_tokens // 2} por reseña)")


[0] relevante=False · texto sin sentido
[1] relevante=True · opiniones y valoración del juego

Tokens de entrada: 1260
Tokens de salida:  93
Total:             1353  (≈ 676 por reseña)


### Paso 2.4: filtrado de todas las reseñas por lotes

In [33]:
# 2.4 Límite de tokens por minuto (TPM) del modelo del filtro, leído de las cabeceras de la respuesta
raw = client.chat.completions.with_raw_response.create(
    model=MODELO_FILTRO,
    messages=[{"role": "user", "content": "Di hola."}],
    reasoning_effort="low",
    max_completion_tokens=50,
)

TPM_FILTRO = int(raw.headers.get("x-ratelimit-limit-tokens", 8000))
print(f"Límite del modelo {MODELO_FILTRO}: {TPM_FILTRO} tokens por minuto")
print(f"Peticiones que quedan hoy: {raw.headers.get('x-ratelimit-remaining-requests')}")

# Tamaño del lote: con el recorte, cada reseña son ~375 tokens
TAM_LOTE_FILTRO = 5

def procesar_filtro(df, tam_lote):
    """Pasa todas las reseñas de df por el filtro, por lotes y con pausas. Devuelve un DataFrame con una fila por reseña."""
    resultados = []
    for inicio in range(0, len(df), tam_lote):
        lote = df.iloc[inicio:inicio + tam_lote]
        resenas, uso = con_reintentos(filtrar_lote, lote)
        resultados.extend(r.model_dump() for r in resenas)

        # Pausa proporcional a los tokens gastados: recuperamos TPM tokens cada 60 s
        pausa = 60 * uso.total_tokens / TPM_FILTRO
        print(f"Lote {inicio // tam_lote + 1}: IDs {list(lote.index)} · {uso.total_tokens} tokens · pausa {pausa:.0f} s")
        if inicio + tam_lote < len(df):   # tras el último lote no hace falta esperar
            time.sleep(pausa)

    return pd.DataFrame(resultados).set_index("id").sort_index()

filtro = procesar_filtro(df_top, TAM_LOTE_FILTRO)
filtro


Límite del modelo openai/gpt-oss-20b: 8000 tokens por minuto
Peticiones que quedan hoy: 998
Lote 1: IDs [0, 1, 2, 4, 5] · 2279 tokens · pausa 17 s
Lote 2: IDs [6, 7, 8, 10, 11] · 2176 tokens · pausa 16 s
Lote 3: IDs [12, 13, 14, 15, 16] · 2198 tokens · pausa 16 s
Lote 4: IDs [17, 18, 20, 22, 24] · 2159 tokens · pausa 16 s
Lote 5: IDs [25, 27, 30, 31, 32] · 2200 tokens · pausa 16 s
Lote 6: IDs [33, 34, 35, 36, 37] · 2199 tokens · pausa 16 s
Lote 7: IDs [39, 41, 42, 43, 44] · 2221 tokens · pausa 17 s
Lote 8: IDs [45, 47, 49] · 1600 tokens · pausa 12 s


,motivo,relevante
id,,
0,texto sin sentido,False
1,opiniones detalladas sobre juego,True
2,crítica y valoración del juego,True
4,análisis de jugabilidad y diseño,True
5,opinión sobre mecánicas y historia,True
6,opiniones sobre jugabilidad y lore,True
7,comentarios sobre mecánicas y AI,True
8,crítica de combate y rendimiento,True
10,descripción de experiencia y atmósfera,True


### Paso 2.5: reseñas relevantes y revisión de los descartes

In [34]:
# 2.5 Unimos la decisión del filtro con los datos originales (por el ID)
df_filtrado = df_top.join(filtro)

print("Resultado del filtro:")
print(df_filtrado['relevante'].value_counts())

# DataFrame reducido: solo las reseñas relevantes (es la entrada del paso 3)
df_relevantes = df_filtrado[df_filtrado['relevante']].copy()
print(f"\nReseñas relevantes que pasan al paso 3: {len(df_relevantes)} de {len(df_filtrado)}")

# Revisión de los descartes: ¿tiene sentido lo que ha quitado el modelo?
descartadas = df_filtrado[~df_filtrado['relevante']]
for i, fila in descartadas.iterrows():
    print(f"\n[{i}] {fila['Valoración']} · Motivo: {fila['motivo']}")
    print(f"     Texto: {fila['Contenido'][:200]}")


Resultado del filtro:
relevante
True     37
False     1
Name: count, dtype: int64

Reseñas relevantes que pasan al paso 3: 37 de 38

[0] Recomendado · Motivo: texto sin sentido
     Texto: suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii


**Resultado del paso 2:** de las 50 reseñas iniciales, 12 eran duplicados exactos (quitados con pandas) y el filtro del LLM descartó 1 más (la [0], spam "suiiii…"). **Pasan 37 reseñas al paso 3.** El filtrado se hizo en 8 lotes de 5 reseñas (~2.200 tokens por llamada, un 27 % del límite de 8.000 tokens por minuto), con pausas calculadas de 12-17 s y sin ningún reintento. El recorte a 1.500 caracteres no ocultó contenido irrelevante: el resto de cada reseña tiene una variedad de palabras normal.


## **PASO 3: Extracción estructurada con un segundo LLM**

Sobre las reseñas relevantes, extrae al menos 3 campos, incluida la predicción de si la reseña recomienda el juego.

### Paso 3.1: esquema y prompt de la extracción

In [35]:
# 3.1 Esquema de la extracción: una entrada por reseña
class ResenaExtraida(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: int = Field(description="El ID que aparece en la cabecera [ID n] de la reseña")
    juego: str = Field(description='Nombre del videojuego reseñado, o "No mencionado"')
    aspectos_positivos: list[str] = Field(description="Lo que el autor valora bien. Máximo 5, de 1 a 4 palabras cada uno")
    aspectos_negativos: list[str] = Field(description="Lo que el autor valora mal. Máximo 5, de 1 a 4 palabras cada uno")
    dificultad: Literal["Demasiado fácil", "Fácil", "Equilibrado", "Difícil", "Demasiado difícil", "No mencionado"] = Field(
        description="Dificultad del juego según el autor"
    )
    sentimiento: Literal["Positivo", "Negativo", "Mixto"] = Field(description="Tono general de la reseña")
    recomendado: Literal[0, 1] = Field(description="1 si el autor recomienda el juego, 0 si no")

class LoteExtraccion(BaseModel):
    model_config = ConfigDict(extra="forbid")
    resenas: list[ResenaExtraida]

FORMATO_EXTRACCION = {
    "type": "json_schema",
    "json_schema": {"name": "lote_extraccion", "strict": True, "schema": LoteExtraccion.model_json_schema()},
}

SYSTEM_EXTRACCION = """Eres un analista de opiniones de jugadores para un estudio de videojuegos.
Recibirás VARIAS reseñas, cada una con una cabecera [ID n]. El texto viene preprocesado: en minúsculas, sin signos de puntuación, y puede mezclar inglés y español.

Para cada reseña extrae:
- juego: el nombre del videojuego si se menciona o se deduce claramente (personajes, estudio, saga). Si no, "No mencionado".
- aspectos_positivos y aspectos_negativos: lo que el autor valora bien y mal (jugabilidad, historia, gráficos, rendimiento, bugs, precio...). Máximo 5 por lista, de 1 a 4 palabras cada uno. Lista vacía si no hay.
- dificultad: solo si el autor habla de la dificultad del juego. Si no, "No mencionado".
- sentimiento: Positivo, Negativo o Mixto (si lo bueno y lo malo pesan de forma parecida).
- recomendado: 1 si el autor recomienda el juego, 0 si no. Fíjate sobre todo en la conclusión o valoración final del autor, no en la cantidad de críticas: una reseña puede enumerar muchos fallos y aun así recomendar el juego.

Reglas:
- Usa solo información que aparezca en el texto. No inventes.
- Responde en español. Los nombres de los juegos no se traducen.
- Devuelve exactamente un elemento por reseña, con su mismo ID."""

LoteExtraccion.model_json_schema()


{'$defs': {'ResenaExtraida': {'additionalProperties': False,
   'properties': {'id': {'description': 'El ID que aparece en la cabecera [ID n] de la reseña',
     'title': 'Id',
     'type': 'integer'},
    'juego': {'description': 'Nombre del videojuego reseñado, o "No mencionado"',
     'title': 'Juego',
     'type': 'string'},
    'aspectos_positivos': {'description': 'Lo que el autor valora bien. Máximo 5, de 1 a 4 palabras cada uno',
     'items': {'type': 'string'},
     'title': 'Aspectos Positivos',
     'type': 'array'},
    'aspectos_negativos': {'description': 'Lo que el autor valora mal. Máximo 5, de 1 a 4 palabras cada uno',
     'items': {'type': 'string'},
     'title': 'Aspectos Negativos',
     'type': 'array'},
    'dificultad': {'description': 'Dificultad del juego según el autor',
     'enum': ['Demasiado fácil',
      'Fácil',
      'Equilibrado',
      'Difícil',
      'Demasiado difícil',
      'No mencionado'],
     'title': 'Dificultad',
     'type': 'string'},


### Paso 3.2: llamada al modelo y prueba con 2 reseñas

In [36]:
# 3.2 Función de extracción: igual que filtrar_lote, pero con el modelo, prompt y esquema de la extracción
def extraer_lote(filas):
    """Envía un lote de reseñas al modelo de extracción. Devuelve (lista de ResenaExtraida, usage)."""
    response = client.chat.completions.create(
        model=MODELO_EXTRACCION,
        messages=[
            {"role": "system", "content": SYSTEM_EXTRACCION},
            {"role": "user", "content": texto_lote(filas, MAX_CARACTERES_EXTRACCION)},  # texto entero (None)
        ],
        temperature=0,
        reasoning_effort="low",
        max_completion_tokens=800 + 400 * len(filas),   # salida más larga que en el filtro: listas de aspectos
        response_format=FORMATO_EXTRACCION,
    )

    if response.choices[0].finish_reason == "length":
        raise ValueError("Respuesta cortada: sube max_completion_tokens o reduce el lote")

    lote = LoteExtraccion.model_validate_json(response.choices[0].message.content)

    ids_enviados = sorted(int(i) for i in filas.index)
    ids_recibidos = sorted(r.id for r in lote.resenas)
    if ids_enviados != ids_recibidos:
        raise ValueError(f"IDs enviados {ids_enviados} y recibidos {ids_recibidos} no coinciden")

    return lote.resenas, response.usage


# Prueba con las 2 primeras reseñas relevantes
prueba = df_relevantes.head(2)
extraidas, uso = con_reintentos(extraer_lote, prueba)

pd.DataFrame([r.model_dump() for r in extraidas]).set_index("id")


,juego,aspectos_positivos,aspectos_negativos,dificultad,sentimiento,recomendado
id,,,,,,
1,Cyberpunk 2077,"[diseño urbano, detalle vertical, libertad exp...","[bugs menores, demasiado fácil, rendimiento es...",Difícil,Positivo,1
2,Cyberpunk 2077,"[gráficos sólidos, atmósfera cyberpunk, banda ...","[bugs frecuentes, personalización limitada, mi...",No mencionado,Negativo,0


In [37]:
print(f"Tokens de entrada: {uso.prompt_tokens}")
print(f"Tokens de salida:  {uso.completion_tokens}")
print(f"Total:             {uso.total_tokens}  (≈ {uso.total_tokens // len(prueba)} por reseña)")

# Solo para comprobar a ojo (el modelo NO ha visto esta columna)
prueba[['Valoración', 'Recomendado_binario']]


Tokens de entrada: 3793
Tokens de salida:  549
Total:             4342  (≈ 2171 por reseña)


,Valoración,Recomendado_binario
id,,
1,Recomendado,1
2,No recomendado,0


### Paso 3.3: extracción de todas las reseñas por lotes

In [ ]:
# 3.3 Límite de tokens por minuto del modelo de extracción
raw = client.chat.completions.with_raw_response.create(
    model=MODELO_EXTRACCION,
    messages=[{"role": "user", "content": "Di hola."}],
    reasoning_effort="low",
    max_completion_tokens=50,
)
TPM_EXTRACCION = int(raw.headers.get("x-ratelimit-limit-tokens", 8000))
print(f"Límite del modelo {MODELO_EXTRACCION}: {TPM_EXTRACCION} tokens por minuto")
print(f"Peticiones que quedan hoy: {raw.headers.get('x-ratelimit-remaining-requests')}")

# Con el texto entero, cada reseña son ~2.100 tokens: 2 por lote ≈ 53 % del límite por minuto
TAM_LOTE_EXTRACCION = 2

def procesar_extraccion(df, tam_lote):
    """Pasa todas las reseñas de df por la extracción, por lotes y con pausas. Devuelve un DataFrame con una fila por reseña."""
    resultados = []
    for inicio in range(0, len(df), tam_lote):
        lote = df.iloc[inicio:inicio + tam_lote]
        extraidas, uso = con_reintentos(extraer_lote, lote)
        resultados.extend(r.model_dump() for r in extraidas)

        pausa = 60 * uso.total_tokens / TPM_EXTRACCION
        print(f"Lote {inicio // tam_lote + 1}: IDs {list(lote.index)} · {uso.total_tokens} tokens · pausa {pausa:.0f} s")
        if inicio + tam_lote < len(df):
            time.sleep(pausa)

    return pd.DataFrame(resultados).set_index("id").sort_index()

extraccion = procesar_extraccion(df_relevantes, TAM_LOTE_EXTRACCION)
extraccion


Límite del modelo openai/gpt-oss-120b: 8000 tokens por minuto
Peticiones que quedan hoy: 998
   429: esperamos 4 s (intento 1 de 5)
Lote 1: IDs [1, 2] · 4338 tokens · pausa 33 s
Lote 2: IDs [4, 5] · 4266 tokens · pausa 32 s
Lote 3: IDs [6, 7] · 4106 tokens · pausa 31 s
Lote 4: IDs [8, 10] · 4297 tokens · pausa 32 s
Lote 5: IDs [11, 12] · 4286 tokens · pausa 32 s
Lote 6: IDs [13, 14] · 4411 tokens · pausa 33 s
Lote 7: IDs [15, 16] · 4332 tokens · pausa 32 s
Lote 8: IDs [17, 18] · 4189 tokens · pausa 31 s
Lote 9: IDs [20, 22] · 4117 tokens · pausa 31 s
Lote 10: IDs [24, 25] · 4153 tokens · pausa 31 s
Lote 11: IDs [27, 30] · 4275 tokens · pausa 32 s


### Paso 3.4: resultado final de la extracción

In [ ]:
# 3.4 Unimos la extracción con los datos originales (por el ID) y guardamos el resultado
resultado = df_relevantes[['Contenido', 'Valoración', 'Recomendado_binario']].join(extraccion)
resultado.to_csv('resultado_extraccion.csv')
print(f"Resultado final: {resultado.shape[0]} reseñas x {resultado.shape[1]} columnas → guardado en resultado_extraccion.csv")

# Resumen de lo extraído
print("\nJuegos:");      print(resultado['juego'].value_counts())
print("\nSentimiento:"); print(resultado['sentimiento'].value_counts())
print("\nDificultad:");  print(resultado['dificultad'].value_counts())
print("\nRecomendado (predicción del LLM):"); print(resultado['recomendado'].value_counts())


Resultado final: 37 reseñas x 9 columnas → guardado en resultado_extraccion.csv

Juegos:
juego
Cyberpunk 2077                                 28
Elden Ring                                      8
The Elder Scrolls V: Skyrim Special Edition     1
Name: count, dtype: int64

Sentimiento:
sentimiento
Mixto       26
Negativo     8
Positivo     3
Name: count, dtype: int64

Dificultad:
dificultad
No mencionado        28
Difícil               6
Demasiado difícil     1
Fácil                 1
Equilibrado           1
Name: count, dtype: int64

Recomendado (predicción del LLM):
recomendado
0    22
1    15
Name: count, dtype: int64


## **PASO 4: Evaluación del prompt**

Compara la predicción del LLM con `Recomendado_binario`: accuracy, tabla cruzada y análisis de los fallos.

4.1: accuracy y tablas cruzadas

In [ ]:
# 4.1 Comparamos la predicción del LLM con la valoración real (que el modelo no ha visto)
aciertos = resultado['recomendado'] == resultado['Recomendado_binario']
print(f"Accuracy: {aciertos.mean():.1%} ({aciertos.sum()} aciertos de {len(aciertos)})")

# Tabla cruzada: filas = valoración real, columnas = predicción del LLM
display(pd.crosstab(resultado['Recomendado_binario'], resultado['recomendado'],
                    rownames=['Real'], colnames=['Predicción LLM'], margins=True))

# ¿Dónde se concentran los fallos? Sentimiento extraído frente a acierto
display(pd.crosstab(resultado['sentimiento'], aciertos.map({True: 'Acierto', False: 'Fallo'})))


Accuracy: 89.2% (33 aciertos de 37)


Predicción LLM,0,1,All
Real,,,
0,19,1,20
1,3,14,17
All,22,15,37


col_0,Acierto,Fallo
sentimiento,,
Mixto,22,4
Negativo,8,0
Positivo,3,0


4.2: análisis de los fallos

In [ ]:
# 4.2 Revisamos cada fallo: ¿se equivoca el modelo o la reseña es ambigua?
fallos = resultado[~aciertos]
for i, fila in fallos.iterrows():
    print(f"[{i}] {fila['juego']} · Real: {fila['Recomendado_binario']} · LLM: {fila['recomendado']} · Sentimiento: {fila['sentimiento']}")
    print(f"   Positivos: {fila['aspectos_positivos']}")
    print(f"   Negativos: {fila['aspectos_negativos']}")
    print(f"   Final de la reseña: ...{fila['Contenido'][-300:]}\n")


[14] Elden Ring · Real: 1 · LLM: 0 · Sentimiento: Mixto
   Positivos: ['alta calidad', 'combat timing', 'mundo atmosférico', 'diseño boss', 'estética visual']
   Negativos: ['inputs laggy', 'UI confusa', 'secretos crípticos', 'progresión pobre', 'historia incomprensible']
   Final de la reseña: ...second walk away there is reason it couldnt have been right next to the boss door it didnt add any challenge all it did was make the boss take longer quality of lifetldr if you just want to git good at fighting big bosses in pretty landscape this game is for you otherwise maybe dont believe the hype

[35] Cyberpunk 2077 · Real: 1 · LLM: 0 · Sentimiento: Mixto
   Positivos: ['libertad exploración', 'ciudad vibrante', 'personajes interesantes', 'árbol habilidades', 'momentos inmersivos']
   Negativos: ['bugs frecuentes', 'marketing exagerado', 'historia rígida', 'rendimiento inestable', 'sistema RPG pobre']
   Final de la reseña: ...er that exists in an open world it grants the player the freed

## **Justificación de decisiones**

Explica aquí, con tus palabras:

- Qué modelo usas en cada paso y por qué

He utilizado Groq, por la carga gratuita pero tambien tengo la Google studio API lista (limitada tambien)

`openai/gpt-oss-20b`. Decidir si una reseña es relevante es una tarea sencilla (sí/no)
`openai/gpt-oss-120b`.Extraer varios campos/variables y, sobre todo, deducir si el autor recomienda el juego en reseñas largas y con matices es una tarea más difícil, que pide un modelo más capaz.

De esta forma tambien ahorramos costo adaptando el modelo a cada tarea

- Qué límites tiene y cómo los respetas: tamaño de los lotes, pausas, reintentos y qué pasa si se alcanza el límite diario

Groq; 8.000 TPR en cada modelo y 1.000 peticiones al día.
Tamaño de los lotes: calculado a partir de una llamada de prueba con 2 reseñas

Filtros, he intentado analizar y recortar las resenas, haciendo pruebas y dando al modelo los 1500 primeros caracteres para que clasifique o sino la resena enetera, dando los 1500 primeros caracteres se reducia el numeor de tokens a 1/5 parte. Pero al final he considerado dejar la resena enetera, asi qeu consume unos 2200 tokens que siendo 8000 tokens el limite nos quedarmos en 3 resenas por lote (4 quedaría muy justo).

Por un lado, el TPM se gestiona con esperas/pausas entre lotes, por lo que nunca se llega y el limite por día esta muy lejos, hemos lanzado unas 30 llamadas de 1000 (hemos analizado 50 resenas solo)

Si se alcanzara el límite del modelo, podriamos cambiar al a otro sin problema.

- Si recortas el texto de las reseñas en algún paso, cuánto y por qué

Ene modelo 1 si se ha planteado recortar a 1500 para saber si la resena es spam o tienen opiniones.

Para el modelo 2, el texto completo dado que necesitamos mas información de cada resena.

Antes del LLM se quitaron 12 duplicados exactos con pandas**: comparar textos idénticos es una operación exacta que no necesita un LLM y así no se gastan tokens.

- Qué técnicas de prompting has aplicado

Rol** en el mensaje de sistema ("moderador de contenido", "analista de opiniones de jugadores").

Contexto del dato:** se avisa de que el texto está en minúsculas, sin puntuación y mezclando inglés y español, para que no se descarte por estar "mal escrito".

Criterios explícitos** de qué es y qué no es relevante, con un ejemplo concreto ("suiiiii").

Reglas para los casos dudosos:** "en caso de duda, relevante" (perder una reseña útil cuesta más que dejar pasar una regular); "una reseña puede enumerar muchos fallos y aun así recomendar el juego".

Delimitadores e identificadores:** cada reseña va con su cabecera `[ID n]` y separada por `---`, para que el modelo no mezcle reseñas y para comprobar que vuelven todas.

Salida estructurada con esquema estricto (Pydantic + JSON Schema)**, con opciones cerradas (Literal) en sentimiento, dificultad y recomendado, y validación** de cada respuesta

Zero-shot:** no se añadieron ejemplos para no encarecer cada llamada


- Qué conclusiones sacas de la evaluación

Accuracy del 89,2 % (33 de 37)**, frente al 54 % de predecir siempre la clase mayoritaria.

Los 4 fallos son reseñas "Mixto"**, y 3 de ellos son reseñas que recomiendan el juego y el modelo predijo que no. Tras revisarlos, **3 se deben a reseñas ambiguas** (recomendaciones condicionales o una nota de 6/10) y **1 a un error del modelo**, que ignoró la frase explícita "dejo una reseña negativa".

Otros problemas observados:** el modelo no siempre respeta la instrucción de idioma (algunos aspectos salen en inglés) y en un caso se contradice (aspecto negativo "demasiada facilidad" con dificultad "Difícil").

Mejoras posibles:** pedir que se dé prioridad a las señales explícitas (frases de recomendación, notas numéricas), reforzar la instrucción de idioma y añadir un ejemplo few-shot de reseña mixta; después, volver a medir para comparar.

Limitación:** 37 reseñas, casi todas de *Cyberpunk 2077*, es una muestra pequeña: el accuracy es orientativo.